# Geospatial Image Stitching & Analysis - Final Submission
This notebook reconstructs a large map from overlapping patches and answers MCQs using InternVL2.

In [ ]:
import os, cv2, numpy as np, pandas as pd, re, torch
from PIL import Image

# ---------------- STITCHER LOGIC ----------------
class MapStitcher:
    def __init__(self, patch_dir, grid_size=(15, 15)):
        self.patch_dir = patch_dir
        self.grid_size = grid_size
        self.patches = self.load_patches()
        self.grid = [[None for _ in range(15)] for _ in range(15)]
        self.used = set()

    def load_patches(self):
        p = {}
        for f in os.listdir(self.patch_dir):
            if f.endswith('.png'):
                idx = int(re.findall(r'\d+', f)[0])
                p[idx] = cv2.imread(f'{self.patch_dir}/{f}')
        return p

    def match(self, left=None, top=None):
        best = (1e9, None)
        for idx, img_orig in self.patches.items():
            if idx in self.used: continue
            for rot in [0,1,2,3]:
                img = np.rot90(img_orig, rot)
                dx, dy = 0, 0
                if left is not None: dx = np.mean(np.abs(left[:, -32:] - img[:, :32]))
                if top is not None: dy = np.mean(np.abs(top[-32:, :] - img[:32, :]))
                if dx + dy < best[0]: 
                    best = (dx + dy, (idx, rot, 32, 32))
                    if dx + dy < 0.5: return best[1]
        return best[1]

    def run(self):
        self.grid[0][0] = (0, 0, 0, 0)
        self.used.add(0)
        for r in range(15):
            for c in range(15):
                if r==0 and c==0: continue
                l = np.rot90(self.patches[self.grid[r][c-1][0]], self.grid[r][c-1][1]) if c>0 else None
                t = np.rot90(self.patches[self.grid[r-1][c][0]], self.grid[r-1][c][1]) if r>0 else None
                self.grid[r][c] = self.match(l, t)
                self.used.add(self.grid[r][c][0])
        
        canvas = np.zeros((1500, 1500, 3), dtype=np.uint8)
        for r in range(15):
            for c in range(15):
                idx, rot, ox, oy = self.grid[r][c]
                y, x = r*96, c*96
                canvas[y:y+128, x:x+128] = np.rot90(self.patches[idx], rot)
        return canvas

# ---------------- MAIN EXECUTION ----------------
stitcher = MapStitcher('patches')
map_img = stitcher.run()
cv2.imwrite('map.png', map_img)

# Add VLM Loader and MCQ logic here... (InternVL2 placeholder)
